In [1]:
from datasets import load_dataset
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainingArguments, Seq2SeqTrainer
from huggingface_hub import login

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import os
login(token=os.environ["HF_TOKEN"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

main_seed=42

model_name = "facebook/mbart-large-50"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

tokenizer.src_lang = "ru_RU"
tokenizer.tgt_lang = "ru_RU"

dataset = load_dataset(
    "csv", 
    data_files={
        "train": "seq2seq_dataset/train.csv",
        "test": "seq2seq_dataset/test.csv", 
        "val": "seq2seq_dataset/val.csv", 
        },
    delimiter="\t"
)

/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocess(batch):
    model_inputs = tokenizer(
        batch["whisper_out"],
        max_length=225,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["label"],
            max_length=225,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess,
    batched=True,
    num_proc=2,
    remove_columns=dataset["train"].column_names
)

In [3]:
import evaluate
import re
import unicodedata

def normalize(text: str) -> str:
    text = text.lower()
    
    normalized_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat == "Zs":
            normalized_chars.append(ch)
        elif unicodedata.combining(ch) and ch in "̄":
            normalized_chars.append(ch)
        elif ch in ["ӈ"]:
            normalized_chars.append(ch)

    return re.sub(r"\s+", " ", "".join(normalized_chars)).strip()

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    orh_wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    orh_cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)

    pred_str_norm = [normalize(pred) for pred in pred_str]
    label_str_norm = [normalize(label) for label in label_str]

    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [ 
        label_str_norm[i] for i in range(len(label_str_norm))if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric_wer.compute(predictions=pred_str_norm, references=label_str_norm)
    cer = 100 * metric_cer.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer": wer, "cer": cer, "orh_wer": orh_wer, "orh_cer": orh_cer}

In [4]:
args = Seq2SeqTrainingArguments(
    output_dir="./mbart_correction_mansi_v1",
    overwrite_output_dir=True,

    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    fp16=False,
    fp16_full_eval=False,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,


    num_train_epochs=20,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    
    seed=main_seed,
    data_seed=main_seed,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=False,
    hub_private_repo=False,
    hub_always_push=False,
    optim="adamw_torch",
    predict_with_generate=True,
    generation_max_length=225
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)
trainer.train()

trainer.save_model("./mbart_correction_mansi_v1")

/tmp/ipykernel_489219/1747892733.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss,Validation Loss,Wer,Cer,Orh Wer,Orh Cer
2000,0.089400,0.067999,18.538426,5.458803,23.716459,6.479112
4000,0.073300,0.062366,18.919304,5.755610,23.850271,6.735518
6000,0.062900,0.060814,19.670381,6.049231,24.797521,7.048620
8000,0.057800,0.057705,20.375182,6.194183,25.674343,7.216663
10000,0.046500,0.057035,18.990496,6.170290,24.131981,7.144645
12000,0.047400,0.055882,19.745132,6.246748,24.910205,7.229433
14000,0.046100,0.055310,19.346456,6.252588,24.114374,7.188571
16000,0.037000,0.057273,19.132880,6.085867,24.212973,7.067519
18000,0.037800,0.058016,19.253907,6.152768,24.610888,7.177845
20000,0.027000,0.060605,19.499519,6.237722,24.825692,7.269273


/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
